In [1]:
import os
import json
import math
import gc
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from PIL import Image
import numpy as np

# ---------------------------------------------------------------------------
# FULL RUN CONFIGURATION (Actual Training)
# ---------------------------------------------------------------------------
CFG = {
    # Paths (Kaggle working directory)
    "working_dir": "/kaggle/working",
    "activations_path": "/kaggle/working/activations_full.pt",
    "sae_weights_path": "/kaggle/working/micro_sae_1024d_full.pt",
    "sae_config_path": "/kaggle/working/config_full.json",

    # Model
    "model_id": "llava-hf/llava-1.5-7b-hf",

    # Dataset sizes -- FULL SCALE
    "coco_num_images": 10_000,        # Broad baseline for diverse features
    "concept_num_images": 500,        # 500 zebras, 500 fire trucks

    # Activation extraction
    "vision_hidden_dim": 1024,        
    "num_patches": 576,               
    "extraction_batch_size": 16,      # Keeps VRAM safe during extraction
    "penultimate_layer_idx": -2,      
    "save_every_n_batches": 200,      # Flushes to disk every ~3200 images to prevent RAM crashes

    # SAE architecture
    "sae_input_dim": 1024,
    "sae_dict_size": 4096,            # 4x expansion factor
    "sae_topk": None,                 

    # SAE training
    "sae_epochs": 2,                  # 2 passes over ~12 million tokens is plenty for low-level features
    "sae_batch_size": 4096,           # Large batch size for fast stable gradient updates
    "sae_lr": 3e-4,                   # Slightly lower LR for stability on the large dataset
    "sae_l1_coeff": 1e-3,             

    # Hub upload
    "hf_repo_name": "llava-micro-sae", # The final production repository name
}

# Ensure output directory exists (no-op on Kaggle, useful locally)
os.makedirs(CFG["working_dir"], exist_ok=True)

In [2]:
# Hugging Face Authentication
def step1_authenticate():
    """
    Retrieve HF_TOKEN from Kaggle secrets and log in.
    Falls back to environment variable HF_TOKEN if not running on Kaggle.
    """
    print("=" * 70)
    print("STEP 1: Hugging Face Authentication")
    print("=" * 70)

    token = None

    # --- Try Kaggle secrets first ---
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        token = user_secrets.get_secret("HF_TOKEN")
        print("[OK] Retrieved HF_TOKEN from Kaggle secrets.")
    except Exception as e:
        print(f"[WARN] Kaggle secrets unavailable ({e}). Trying env variable...")
        token = os.environ.get("HF_TOKEN", None)

    if token is None:
        raise RuntimeError(
            "No HF_TOKEN found. Set it as a Kaggle secret or env variable."
        )

    from huggingface_hub import login
    login(token=token)
    print("[OK] Logged in to Hugging Face Hub.\n")
    return token

In [3]:
# Dataset Curation (Disk-Streaming + Kaggle Native ImageNet)

import shutil

class DiskImageDataset(Dataset):
    """Reads images one-by-one from local disk to save CPU RAM."""
    def __init__(self, image_dir: Path, transform):
        self.image_paths = list(image_dir.glob("*.jpg"))
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        with Image.open(self.image_paths[idx]) as img:
            img_rgb = img.convert("RGB")
            tensor = self.transform(img_rgb)
        return tensor


def step2_curate_dataset(processor):
    print("=" * 70)
    print("STEP 2: Dataset Curation (Hybrid Local/Streaming Mode)")
    print("=" * 70)

    from datasets import load_dataset

    img_dir = Path(CFG["working_dir"]) / "images"
    if img_dir.exists():
        shutil.rmtree(img_dir)  
    img_dir.mkdir(parents=True, exist_ok=True)

    def clip_transform(pil_img: Image.Image) -> torch.Tensor:
        out = processor.image_processor(images=pil_img, return_tensors="pt")
        return out["pixel_values"].squeeze(0)

    # --- 2a. COCO images (Streaming - this worked perfectly in your last run) ---
    print(f"[...] Streaming and saving {CFG['coco_num_images']} COCO images to disk...")
    try:
        ds_stream = load_dataset("detection-datasets/coco", split="train", streaming=True)
        count = 0
        for ex in ds_stream:
            if count >= CFG["coco_num_images"]: break
            try:
                img = ex["image"]
                if not isinstance(img, Image.Image):
                    img = Image.fromarray(np.array(img))
                
                img_resized = img.convert("RGB").resize((336, 336), Image.Resampling.LANCZOS)
                img_resized.save(img_dir / f"coco_{count}.jpg", quality=85)
                
                img.close()
                img_resized.close()
                del img, img_resized, ex
            except Exception:
                continue
            
            count += 1
            if count % 1000 == 0:
                gc.collect()
                print(f"   ... {count}/{CFG['coco_num_images']} saved")
        
        gc.collect()
        print(f"[OK] COCO: Saved {count} images.")
    except Exception as e:
        print(f"[FAIL] COCO download failed: {e}")

    # --- 2b & 2c. Concept Images (Direct from Kaggle Native ImageNet) ---
    def fetch_local_imagenet(wnid, name, max_count):
        print(f"[...] Fetching {max_count} '{name}' images from local Kaggle ImageNet...")
        # Path to the mounted Kaggle Competition Dataset
        source_dir = Path(f"/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train/{wnid}")
        
        if not source_dir.exists():
            print(f"[FAIL] Directory not found: {source_dir}")
            print("      Did you click 'Add Data' and attach 'imagenet-object-localization-challenge'?")
            return 0
            
        jpeg_files = list(source_dir.glob("*.JPEG"))
        
        count = 0
        for file_path in jpeg_files:
            if count >= max_count: break
            try:
                with Image.open(file_path) as img:
                    img_resized = img.convert("RGB").resize((336, 336), Image.Resampling.LANCZOS)
                    img_resized.save(img_dir / f"{name}_{count}.jpg", quality=85)
                    img_resized.close()
                count += 1
            except Exception:
                continue
                
        print(f"[OK] {name}: Saved {count} images locally.")
        return count

    # n02391049 is the exact WordNet ID for Zebra
    fetch_local_imagenet("n02391049", "zebra", CFG["concept_num_images"])
    
    # n03345487 is the exact WordNet ID for Fire engine / Fire truck
    fetch_local_imagenet("n03345487", "firetruck", CFG["concept_num_images"])

    # --- Create DataLoader reading strictly from Disk ---
    disk_dataset = DiskImageDataset(img_dir, transform=clip_transform)
    print(f"\n[OK] Total images securely stored on disk: {len(disk_dataset)}")

    loader = DataLoader(
        disk_dataset,
        batch_size=CFG["extraction_batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=False,
    )
    print(f"[OK] DataLoader ready ({len(loader)} batches).\n")
    return loader

In [4]:
# Activation Extraction
def step3_extract_activations(loader):
    print("=" * 70)
    print("STEP 3: Activation Extraction")
    print("=" * 70)

    from transformers import LlavaForConditionalGeneration

    print("[...] Loading LLaVA model (vision tower only will be used)...")
    model = LlavaForConditionalGeneration.from_pretrained(
        CFG["model_id"],
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()
    print("[OK] Model loaded.\n")

    vision_tower = model.model.vision_tower

    chunk_size = CFG["save_every_n_batches"]
    chunk_buffer = []
    chunk_idx = 0
    total_patches = 0
    chunk_dir = Path(CFG["working_dir"]) / "activation_chunks"
    chunk_dir.mkdir(parents=True, exist_ok=True)

    print("[...] Extracting patch activations from CLIP Layer 23...")
    
    # 1. THE EXTRACTION LOOP (Reads JPEGs from disk)
    with torch.no_grad():
        for batch_idx, pixel_values in enumerate(loader):
            pixel_values = pixel_values.to(
                device=next(vision_tower.parameters()).device,
                dtype=torch.float16,
            )

            vision_outputs = vision_tower(
                pixel_values,
                output_hidden_states=True,
            )

            hidden = vision_outputs.hidden_states[CFG["penultimate_layer_idx"]]
            patch_tokens = hidden[:, 1:, :]  

            flat = patch_tokens.reshape(-1, CFG["vision_hidden_dim"]).cpu().half()
            chunk_buffer.append(flat)
            total_patches += flat.shape[0]

            del vision_outputs, hidden, patch_tokens, flat, pixel_values
            torch.cuda.empty_cache()

            if (batch_idx + 1) % 50 == 0:
                print(f"   Batch {batch_idx + 1}/{len(loader)} -- {total_patches:,} patches so far")

            if (batch_idx + 1) % chunk_size == 0:
                chunk_tensor = torch.cat(chunk_buffer, dim=0)
                chunk_path = chunk_dir / f"chunk_{chunk_idx:04d}.pt"
                torch.save(chunk_tensor, chunk_path)
                print(f"   [SAVE] Chunk {chunk_idx} saved: {chunk_tensor.shape[0]:,} patches -> {chunk_path}")
                
                del chunk_tensor, chunk_buffer
                chunk_buffer = []
                chunk_idx += 1
                gc.collect()

    if chunk_buffer:
        chunk_tensor = torch.cat(chunk_buffer, dim=0)
        chunk_path = chunk_dir / f"chunk_{chunk_idx:04d}.pt"
        torch.save(chunk_tensor, chunk_path)
        print(f"   [SAVE] Final chunk {chunk_idx} saved: {chunk_tensor.shape[0]:,} patches")
        del chunk_tensor, chunk_buffer
        gc.collect()

    print(f"\n[OK] Total patches extracted: {total_patches:,}")

    # 2. DELETE JPEGS TO FREE DISK SPACE 
    # (Safe to do now because DataLoader is finished!)
    img_dir = Path(CFG["working_dir"]) / "images"
    if img_dir.exists():
        import shutil
        shutil.rmtree(img_dir)
        print("[OK] Deleted original JPEG images to free disk space.")

    # 3. DELETE LLAVA MODEL TO FREE RAM/GPU
    del model, vision_tower
    gc.collect()
    torch.cuda.empty_cache()
    print("[OK] LLaVA model deleted and GPU cache cleared.")

    # 4. PRE-ALLOCATED MERGE (Extremely safe for RAM/Disk limits)
    print("[...] Merging activation chunks into single file...")
    chunk_files = sorted(chunk_dir.glob("chunk_*.pt"))
    
    total_rows = 0
    for cf in chunk_files:
        c = torch.load(cf, map_location="cpu")
        total_rows += c.shape[0]
        del c
    gc.collect()
    print(f"   Total rows to merge: {total_rows:,}")
    
    activations_tensor = torch.empty((total_rows, CFG["vision_hidden_dim"]), dtype=torch.float16)
    offset = 0
    for cf in chunk_files:
        c = torch.load(cf, map_location="cpu")
        rows = c.shape[0]
        activations_tensor[offset:offset + rows] = c
        offset += rows
        cf.unlink()  # delete chunk IMMEDIATELY to free disk space
        del c
        gc.collect()
    
    print(f"[OK] Final activation tensor: {activations_tensor.shape} ({activations_tensor.dtype})")
    torch.save(activations_tensor, CFG["activations_path"])
    print(f"[SAVE] Saved to {CFG['activations_path']}")

    chunk_dir.rmdir()
    del activations_tensor
    gc.collect()
    print("[OK] Activation extraction complete.\n")

In [5]:
# Train the Micro-SAE

class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim: int, dict_size: int):
        super().__init__()
        self.input_dim = input_dim
        self.dict_size = dict_size

        self.encoder = nn.Linear(input_dim, dict_size)
        self.decoder = nn.Linear(dict_size, input_dim, bias=True)

        nn.init.xavier_uniform_(self.encoder.weight)
        nn.init.zeros_(self.encoder.bias)
        nn.init.xavier_uniform_(self.decoder.weight)
        nn.init.zeros_(self.decoder.bias)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(self.encoder(x))

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    def forward(self, x: torch.Tensor):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

    @torch.no_grad()
    def normalize_decoder(self):
        norms = self.decoder.weight.norm(dim=0, keepdim=True).clamp(min=1e-8)
        self.decoder.weight.div_(norms)


class TopKSparseAutoencoder(nn.Module):
    def __init__(self, input_dim: int, dict_size: int, k: int = 32):
        super().__init__()
        self.input_dim = input_dim
        self.dict_size = dict_size
        self.k = k

        self.encoder = nn.Linear(input_dim, dict_size)
        self.decoder = nn.Linear(dict_size, input_dim, bias=True)

        nn.init.xavier_uniform_(self.encoder.weight)
        nn.init.zeros_(self.encoder.bias)
        nn.init.xavier_uniform_(self.decoder.weight)
        nn.init.zeros_(self.decoder.bias)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        pre_act = self.encoder(x)
        topk_vals, topk_idx = pre_act.topk(self.k, dim=-1)
        z = torch.zeros_like(pre_act)
        z.scatter_(-1, topk_idx, F.relu(topk_vals))
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    def forward(self, x: torch.Tensor):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

    @torch.no_grad()
    def normalize_decoder(self):
        norms = self.decoder.weight.norm(dim=0, keepdim=True).clamp(min=1e-8)
        self.decoder.weight.div_(norms)


class ActivationDataset(Dataset):
    def __init__(self, tensor: torch.Tensor):
        self.data = tensor

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx]


def step4_train_sae():
    print("=" * 70)
    print("STEP 4: Train the Micro-SAE")
    print("=" * 70)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. LOAD DATA IN FLOAT16 (RAM usage: ~13 GB -> Safe!)
    print(f"[...] Loading activations from {CFG['activations_path']}...")
    activations = torch.load(CFG["activations_path"], map_location="cpu")
    print(f"[OK] Loaded {activations.shape[0]:,} activation vectors of dim {activations.shape[1]}.")
    assert activations.shape[1] == CFG["sae_input_dim"]

    # Shuffle once on CPU
    perm = torch.randperm(activations.shape[0])
    activations = activations[perm]

    act_dataset = ActivationDataset(activations)
    act_loader = DataLoader(
        act_dataset,
        batch_size=CFG["sae_batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True,
    )
    print(f"[OK] Training DataLoader: {len(act_loader)} batches per epoch.\n")

    # 2. INIT MODEL
    use_topk = CFG["sae_topk"] is not None
    if use_topk:
        print(f"[...] Using TopK SAE (k={CFG['sae_topk']})")
        sae = TopKSparseAutoencoder(
            input_dim=CFG["sae_input_dim"],
            dict_size=CFG["sae_dict_size"],
            k=CFG["sae_topk"],
        ).to(device)
    else:
        print("[...] Using ReLU SAE with L1 sparsity penalty")
        sae = SparseAutoencoder(
            input_dim=CFG["sae_input_dim"],
            dict_size=CFG["sae_dict_size"],
        ).to(device)

    total_params = sum(p.numel() for p in sae.parameters())
    print(f"[OK] SAE parameters: {total_params:,}")

    optimizer = torch.optim.Adam(sae.parameters(), lr=CFG["sae_lr"])

    # 3. TRAINING LOOP
    print("[...] Starting training...\n")
    for epoch in range(CFG["sae_epochs"]):
        epoch_mse = 0.0
        epoch_l1 = 0.0
        epoch_loss = 0.0
        n_batches = 0

        sae.train()
        for batch_idx, batch in enumerate(act_loader):
            
            # --- THE MEMORY FIX ---
            # Cast the float16 batch to float32 strictly on the GPU!
            batch = batch.to(device).float()

            x_hat, z = sae(batch)
            mse_loss = F.mse_loss(x_hat, batch)

            if use_topk:
                l1_loss = torch.tensor(0.0, device=device)
                loss = mse_loss
            else:
                l1_loss = z.abs().mean()
                loss = mse_loss + CFG["sae_l1_coeff"] * l1_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            sae.normalize_decoder()

            epoch_mse += mse_loss.item()
            epoch_l1 += l1_loss.item()
            epoch_loss += loss.item()
            n_batches += 1

            if (batch_idx + 1) % 200 == 0:
                avg_mse = epoch_mse / n_batches
                avg_l1 = epoch_l1 / n_batches
                with torch.no_grad():
                    l0 = (z > 0).float().sum(dim=-1).mean().item()
                print(
                    f"   Epoch {epoch+1} | Batch {batch_idx+1}/{len(act_loader)} | "
                    f"MSE={avg_mse:.6f} | L1={avg_l1:.6f} | L0={l0:.1f}"
                )

        avg_mse = epoch_mse / n_batches
        avg_l1 = epoch_l1 / n_batches
        avg_loss = epoch_loss / n_batches
        print(
            f"\n   * Epoch {epoch+1}/{CFG['sae_epochs']} complete -- "
            f"Loss={avg_loss:.6f} | MSE={avg_mse:.6f} | L1={avg_l1:.6f}\n"
        )

    # 4. SAVE ARTIFACTS
    print(f"[SAVE] Saving SAE weights to {CFG['sae_weights_path']}...")
    torch.save(sae.state_dict(), CFG["sae_weights_path"])

    config = {
        "architecture": "TopKSparseAutoencoder" if use_topk else "SparseAutoencoder",
        "input_dim": CFG["sae_input_dim"],
        "dict_size": CFG["sae_dict_size"],
        "topk": CFG["sae_topk"],
        "activation": "topk" if use_topk else "relu",
        "source_model": CFG["model_id"],
        "source_layer": "vision_tower.hidden_states[-2] (Layer 23)",
        "num_patches_per_image": CFG["num_patches"],
        "training_epochs": CFG["sae_epochs"],
        "training_lr": CFG["sae_lr"],
        "l1_coeff": CFG["sae_l1_coeff"] if not use_topk else None,
        "total_training_vectors": len(activations),
    }
    with open(CFG["sae_config_path"], "w") as f:
        json.dump(config, f, indent=2)
    print(f"[SAVE] Saved config to {CFG['sae_config_path']}")
    print(f"[OK] SAE training complete.\n")

    del activations, sae
    gc.collect()
    torch.cuda.empty_cache()

In [6]:
# Upload to Hugging Face Hub
def step5_upload_to_hub(token: str):
    """
    Create a HF repo and upload the SAE weights + config.
    """
    print("=" * 70)
    print("STEP 5: Upload to Hugging Face Hub")
    print("=" * 70)

    from huggingface_hub import HfApi

    api = HfApi()

    # Get the authenticated user's username
    user_info = api.whoami(token=token)
    username = user_info["name"]
    repo_id = f"{username}/{CFG['hf_repo_name']}"

    # Create repo (no-op if it already exists)
    print(f"[...] Creating repo: {repo_id}")
    api.create_repo(
        repo_id=repo_id,
        repo_type="model",
        exist_ok=True,
        token=token,
    )
    print(f"[OK] Repo ready: https://huggingface.co/{repo_id}")

    # Upload weights
    print(f"[...] Uploading {CFG['sae_weights_path']}...")
    api.upload_file(
        path_or_fileobj=CFG["sae_weights_path"],
        path_in_repo="micro_sae_1024d.pt",
        repo_id=repo_id,
        token=token,
    )
    print("[OK] Weights uploaded.")

    # Upload config
    print(f"[...] Uploading {CFG['sae_config_path']}...")
    api.upload_file(
        path_or_fileobj=CFG["sae_config_path"],
        path_in_repo="config.json",
        repo_id=repo_id,
        token=token,
    )
    print("[OK] Config uploaded.")
    print(f"\nSUCCESS: All artifacts uploaded to: https://huggingface.co/{repo_id}\n")


In [7]:
print("Micro-SAE Training Pipeline for LLaVA VLM Unlearning")
print("=" * 70)
print(f"   Target model : {CFG['model_id']}")
print(f"   SAE dims     : {CFG['sae_input_dim']} -> {CFG['sae_dict_size']} -> {CFG['sae_input_dim']}")
print(f"   Device       : {'CUDA' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"   GPU          : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("=" * 70 + "\n")

# Authenticate
token = step1_authenticate()

Micro-SAE Training Pipeline for LLaVA VLM Unlearning
   Target model : llava-hf/llava-1.5-7b-hf
   SAE dims     : 1024 -> 4096 -> 1024
   Device       : CUDA
   GPU          : Tesla P100-PCIE-16GB
   VRAM         : 17.1 GB

STEP 1: Hugging Face Authentication
[OK] Retrieved HF_TOKEN from Kaggle secrets.
[OK] Logged in to Hugging Face Hub.



In [8]:
# CLIP Processor
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(CFG["model_id"],use_fast=False)
loader = step2_curate_dataset(processor)
del processor  # not needed after this

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

...

STEP 2: Dataset Curation (Hybrid Local/Streaming Mode)
[...] Streaming and saving 10000 COCO images to disk...


README.md:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

   ... 1000/10000 saved
   ... 2000/10000 saved
   ... 3000/10000 saved
   ... 4000/10000 saved
   ... 5000/10000 saved
   ... 6000/10000 saved
   ... 7000/10000 saved
   ... 8000/10000 saved
   ... 9000/10000 saved
   ... 10000/10000 saved
[OK] COCO: Saved 10000 images.
[...] Fetching 500 'zebra' images from local Kaggle ImageNet...
[OK] zebra: Saved 500 images locally.
[...] Fetching 500 'firetruck' images from local Kaggle ImageNet...
[OK] firetruck: Saved 500 images locally.

[OK] Total images securely stored on disk: 11000
[OK] DataLoader ready (688 batches).



In [9]:
# Extract activations (loads model -> extracts -> frees GPU)
step3_extract_activations(loader)
del loader
gc.collect()

STEP 3: Activation Extraction
[...] Loading LLaVA model (vision tower only will be used)...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

[OK] Model loaded.

[...] Extracting patch activations from CLIP Layer 23...
   Batch 50/688 -- 460,800 patches so far
   Batch 100/688 -- 921,600 patches so far
   Batch 150/688 -- 1,382,400 patches so far
   Batch 200/688 -- 1,843,200 patches so far
   [SAVE] Chunk 0 saved: 1,843,200 patches -> /kaggle/working/activation_chunks/chunk_0000.pt
   Batch 250/688 -- 2,304,000 patches so far
   Batch 300/688 -- 2,764,800 patches so far
   Batch 350/688 -- 3,225,600 patches so far
   Batch 400/688 -- 3,686,400 patches so far
   [SAVE] Chunk 1 saved: 1,843,200 patches -> /kaggle/working/activation_chunks/chunk_0001.pt
   Batch 450/688 -- 4,147,200 patches so far
   Batch 500/688 -- 4,608,000 patches so far
   Batch 550/688 -- 5,068,800 patches so far
   Batch 600/688 -- 5,529,600 patches so far
   [SAVE] Chunk 2 saved: 1,843,200 patches -> /kaggle/working/activation_chunks/chunk_0002.pt
   Batch 650/688 -- 5,990,400 patches so far
   [SAVE] Final chunk 3 saved: 806,400 patches

[OK] Total pa

0

In [10]:
# Train SAE on saved activations
step4_train_sae()


STEP 4: Train the Micro-SAE
[...] Loading activations from /kaggle/working/activations_full.pt...
[OK] Loaded 6,336,000 activation vectors of dim 1024.
[OK] Training DataLoader: 1546 batches per epoch.

[...] Using ReLU SAE with L1 sparsity penalty
[OK] SAE parameters: 8,393,728
[...] Starting training...

   Epoch 1 | Batch 200/1546 | MSE=0.190807 | L1=0.183947 | L0=1924.1
   Epoch 1 | Batch 400/1546 | MSE=0.113673 | L1=0.266832 | L0=2188.4
   Epoch 1 | Batch 600/1546 | MSE=0.080274 | L1=0.321440 | L0=2244.6
   Epoch 1 | Batch 800/1546 | MSE=0.061816 | L1=0.358431 | L0=2256.8
   Epoch 1 | Batch 1000/1546 | MSE=0.050188 | L1=0.384658 | L0=2264.2
   Epoch 1 | Batch 1200/1546 | MSE=0.042245 | L1=0.403795 | L0=2265.9
   Epoch 1 | Batch 1400/1546 | MSE=0.036494 | L1=0.418191 | L0=2267.5

   * Epoch 1/2 complete -- Loss=0.033638 | MSE=0.033211 | L1=0.426554

   Epoch 2 | Batch 200/1546 | MSE=0.001474 | L1=0.507073 | L0=2267.2
   Epoch 2 | Batch 400/1546 | MSE=0.001434 | L1=0.506461 | L0=226

In [11]:
# Upload to Hugging Face Hub
step5_upload_to_hub(token)

print("DONE: Pipeline complete!")

STEP 5: Upload to Hugging Face Hub
[...] Creating repo: AKG2/llava-micro-sae
[OK] Repo ready: https://huggingface.co/AKG2/llava-micro-sae
[...] Uploading /kaggle/working/micro_sae_1024d_full.pt...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[OK] Weights uploaded.
[...] Uploading /kaggle/working/config_full.json...
[OK] Config uploaded.

SUCCESS: All artifacts uploaded to: https://huggingface.co/AKG2/llava-micro-sae

DONE: Pipeline complete!
